# Fase CRISP-DM: Modelado Predictivo y Torneo de Machine Learning
**Plataforma**: AgroData Intelligence Platform (AgroStatsApp)  
**Estándares**: Scikit-Learn Pipelines, TimeSeriesSplit (Respeto de la flecha del tiempo), SWEBOK  
**Propósito**: Entrenar y comparar competitivamente modelos de regresión supervisada (Ridge, Random Forest, HistGradientBoosting) bajo validación cruzada temporal sin fuga de datos, seleccionando el mejor estimador para producción.

In [ ]:
# 1. Configuración de Entorno Resiliente (Google Colab / VS Code / Jupyter Local)
import os
import sys
import subprocess
from pathlib import Path

def setup_environment():
    # A. Detección y preparación automática para Google Colab
    if 'google.colab' in sys.modules or Path('/content').exists():
        print('[INFO] Entorno detectado: Google Colab.')
        repo_dir = Path('/content/Statsfirm')
        if not repo_dir.exists():
            print('[INFO] Clonando repositorio oficial Statsfirm en Colab...')
            subprocess.run(['git', 'clone', 'https://github.com/adansanchezc1-spec/Statsfirm.git', '/content/Statsfirm'], check=True)
        else:
            print('[INFO] Actualizando repositorio en Colab...')
            subprocess.run(['git', '-C', '/content/Statsfirm', 'pull'], check=False)
        
        app_dir = repo_dir / 'AgroStats AndTech' / 'AgroStatsApp'
        if app_dir.exists():
            os.chdir(str(app_dir))
            src_dir = app_dir / 'src'
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            print(f'[OK] Directorio de trabajo establecido en: {app_dir}')
            print(f'[OK] Carpeta src agregada a sys.path: {src_dir}')
            return

    # B. Detección dinámica en Entorno Local (Windows / Linux / WSL / VS Code)
    candidates = [
        Path.cwd() / 'src',
        Path.cwd() / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path.cwd().parent / 'src',
        Path.cwd().parent.parent / 'src',
        Path.cwd().parent.parent / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path(r'c:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\AgroStatsApp\src'),
        Path('/mnt/c/Users/ADAN/OneDrive/Documentos/Statsfirm/AgroStats AndTech/AgroStatsApp/src'),
    ]
    
    curr = Path.cwd().resolve()
    for _ in range(6):
        target = curr / 'AgroStats AndTech' / 'AgroStatsApp' / 'src'
        if target.exists() and (target / 'notebook_code').is_dir():
            candidates.insert(0, target)
            break
        curr = curr.parent

    for c in candidates:
        if c.exists() and (c / 'notebook_code').is_dir():
            resolved = str(c.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            print(f'[OK] Módulo src localizado localmente en: {resolved}')
            return

setup_environment()

# Importación del entrenador de modelos
from notebook_code import ModelTrainer, FeatureEngineer, load_all_raw_datasets
import pandas as pd

print('[OK] Módulo ModelTrainer importado exitosamente.')


In [ ]:
# 2. Carga y Preparación de la Matriz de Características
feat_path = Path('CRISPDM/data/FEATURES/master_feature_matrix.csv')
if feat_path.exists():
    df = pd.read_csv(feat_path)
else:
    # Generación al vuelo si se ejecuta de forma aislada
    raws = load_all_raw_datasets()
    df = FeatureEngineer.create_cyclical_calendar_features(
        FeatureEngineer.create_rolling_features(
            FeatureEngineer.create_lag_features(
                FeatureEngineer.clean_dataset(raws['ideam_pluvio'], date_col='fechaobservacion'),
                target_cols=['valorobservado'],
                lags=[1, 2, 7]
            ),
            target_cols=['valorobservado'],
            windows=[7, 14]
        ),
        date_col='fechaobservacion'
    )
print(f'Matriz de modelado preparada: {df.shape[0]} filas x {df.shape[1]} columnas.')


In [ ]:
# 3. Partición Cronológica Train / Test (80% / 20%)
X_train, y_train, X_test, y_test, train_dates, test_dates = ModelTrainer.prepare_train_test_split(
    df,
    target_col='valorobservado',
    date_col='fechaobservacion',
    test_size=0.2
)
print(f'Entrenamiento: {X_train.shape[0]} registros | Prueba (Hold-out): {X_test.shape[0]} registros')
print(f'Variables predictoras ({X_train.shape[1]}): {list(X_train.columns)}')


In [ ]:
# 4. Validación Cruzada Temporal (TimeSeriesSplit - 5 Folds)
baseline_pipe = ModelTrainer.build_pipeline(model_type='ridge')
cv_results = ModelTrainer.cross_validate_time_series(baseline_pipe, X_train, y_train, n_splits=5)
print('=== RESULTADOS DE VALIDACIÓN CRUZADA TEMPORAL (RIDGE) ===')
print(f'RMSE Promedio: {cv_results["mean_rmse"]:.4f} (± {cv_results["std_rmse"]:.4f})')
print(f'MAE Promedio:  {cv_results["mean_mae"]:.4f}')
print(f'R² Promedio:   {cv_results["mean_r2"]:.4f}')


In [ ]:
# 5. Torneo Competitivo de Algoritmos (Ridge vs Random Forest vs HistGradientBoosting)
winner_pipeline, leaderboard_df, all_models = ModelTrainer.train_competitive_tournament(
    X_train,
    y_train,
    X_test,
    y_test,
    candidate_models=('ridge', 'rf', 'hist_gbr')
)
print('=== TABLA DE CLASIFICACIÓN DEL TORNEO (LEADERBOARD) ===')
display(leaderboard_df)


In [ ]:
# 6. Persistencia del Modelo Ganador en CRISPDM/data/MODEL/
winner_row = leaderboard_df.iloc[0]
model_file, meta_file = ModelTrainer.save_trained_model(
    pipeline=winner_pipeline,
    feature_names=list(X_train.columns),
    metrics=winner_row.to_dict(),
    model_name='best_agro_model'
)
print(f'Modelo ganador ({winner_row["Modelo"]}) persistido en:')
print(f'• Binario:   {model_file}')
print(f'• Metadatos: {meta_file}')
